> # ⚠️ Do not run this notebook unattended
>
> The last section, **`# Hardware validation`**, connects to the real board and uploads the
> package. *Run All* and `jlx run` reach that section.
>
> - **Safe to run on its own:** everything above `# Hardware validation`. It runs on the host,
>   writes the module and its tests, and needs no board.
> - **Hardware section:** run it one cell at a time, reading the **What to do** line above
>   each cell first.

**Where each cell runs.** Everything above `%board connect` runs in ordinary Python
(CPython, the project `.venv`) on this computer. After `%board connect`, cells run in
MicroPython on the board, except cells that start with `%%host`.

Notebook-only: import the package before anything else. Its `__init__` imports `bits`, and
the header cell below rewrites `bits.py`; importing the package first means no later import
reads the half-written file.

In [ ]:
import micropython_i2c  # noqa: F401

# Hardware and configuration

| | Host run | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13 in the micropylab kernel, no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0, USB `/dev/ttyACM0` |
| **I2C bus** | none | none: `bits` is pure arithmetic and never touches the bus |
| **Result** | not run yet on the hardware | not run yet |

# TODO:
Nothing open.

Decided (2026-11-16): `bits.py` moved from hand-written to this notebook. The API and
behaviour are unchanged; the tests are the ones from the hand-written `tests/test_bits.py`
(ported from the old single-file notebook's sections 10.1 to 10.4), one per cell.

# Python file
location: `src/micropython_i2c/bits.py`

tests: `tests/test_bits.py`

## header

In [ ]:
%%extract ../src/micropython_i2c/bits.py
"""``BitTools``: stateless bit and byte helpers.

Sign extension, mask construction and byte assembly get reimplemented in every driver,
each time slightly differently. One implementation means one place to fix.

They are staticmethods on a class rather than module functions so a driver reads as
``self.sign_extend(word, 12)`` next to ``self.reg_read(...)``, and so a part with an
odd format can override one for itself. ``BitTools`` has no class attributes, so mixing
it in adds no obligations; ``RegisterI2CBase`` and ``CRCMixin`` both inherit it.

Bitfields are ``(shift, width)`` tuples, declared once as class constants on a driver.

Generated by ``dev_notebooks/bits_devNB``: edit the notebook, not this file.
"""

# Imports
## External
None: the helpers use only built-in `int` methods (`from_bytes`, `to_bytes`), which
MicroPython and CPython both have.

## Internal
None: `bits` is the bottom of the package; `crc` and `protocol` build on it.

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [ ]:
%%writefile ../tests/test_bits.py
import pytest

from micropython_i2c.bits import BitTools

Notebook-only: the imports the validation and test cells use. `BitTools` itself comes from
the code cell below.

In [ ]:
import pytest

# Intent
**Purpose:** one implementation of the bit and byte arithmetic every driver needs: sign
extension of an $n$-bit two's-complement reading, `(shift, width)` bitfields in a register
word, bit reversal, and big- or little-endian byte assembly.

**Why it exists:** these few lines get rewritten in every driver, each time slightly
differently, and the bugs (sign-extending before shifting out padding, an oversized value
producing a wrong-length transfer) are the same each time. One class means one place to fix.

**How drivers use it:** through inheritance. `RegisterI2CBase` and `CRCMixin` both subclass
`BitTools`, so a driver calls `self.sign_extend(...)`, `self.field_insert(...)`. Every method
is a `staticmethod`, so `BitTools.sign_extend(...)` works unbound too.

```mermaid
flowchart BT
    R[RegisterI2CBase] --> B[BitTools]
    C[CRCMixin] --> B
    D[drivers] --> R
    D --> C
```

**Bitfields** are `(shift, width)` tuples declared once as class constants on a driver, e.g.
the TMP102's conversion-rate field `CR = (6, 2)`: bits 7..6 of the configuration word.

**Not its job:** reading or writing registers (`protocol`), checksums (`crc`).

## What the module defines

| Name | What it is |
|---|---|
| `BitTools` | stateless bit and byte helpers, as staticmethods |
| `BitTools.sign_extend` | read a `bits`-wide value as two's complement |
| `BitTools.field_mask` | aligned mask of a `(shift, width)` field |
| `BitTools.field_extract` | a field's value out of a word |
| `BitTools.field_insert` | a word with one field replaced |
| `BitTools.bit_reverse` | reverse the bit order of a `bits`-wide value |
| `BitTools.bytes_to_int` | bytes to an unsigned int, big- or little-endian |
| `BitTools.int_to_bytes` | an int to `nbytes` bytes, masked to width |

# Code
## TODO:
Nothing

## Public names

In [ ]:
%%extract ../src/micropython_i2c/bits.py -a
__all__ = ["BitTools"]

## `BitTools`
The whole module is this class. The methods do not touch `self` or any class attribute, so
mixing the class into a driver adds nothing the driver must set. Each method masks its
input to the width it works on, so an oversized argument is truncated, never turned into a
wrong-length result.

| Method | Rule |
|---|---|
| `sign_extend(value, bits)` | mask to `bits`, then subtract $2^{bits}$ if the top bit is set |
| `field_mask((shift, width))` | $(2^{width} - 1) \ll shift$ |
| `field_extract(value, field)` | $(value \gg shift)\ \&\ (2^{width} - 1)$ |
| `field_insert(value, field, new)` | clear the field, OR in `new` shifted and masked |
| `bit_reverse(value, bits=8)` | bit $i$ moves to bit $bits - 1 - i$ |
| `bytes_to_int(data, big_endian=True)` | `int.from_bytes` |
| `int_to_bytes(value, nbytes, big_endian=True)` | mask to `8 * nbytes` bits, then `int.to_bytes` |

In [ ]:
%%extract ../src/micropython_i2c/bits.py -a
class BitTools:
    """Pure bit/byte arithmetic. Safe to call unbound: ``BitTools.sign_extend(...)``."""

    @staticmethod
    def sign_extend(value, bits):
        """Interpret ``value`` as a ``bits``-wide two's-complement number.

        ``value`` is masked to width first. On a left-justified part, shift the padding
        out *before* calling this: extending the unshifted word treats the padding
        zeros as magnitude.
        """
        value &= (1 << bits) - 1
        return value - (1 << bits) if value & (1 << (bits - 1)) else value

    @staticmethod
    def field_mask(field):
        """Aligned bitmask for a ``(shift, width)`` field."""
        shift, width = field
        return ((1 << width) - 1) << shift

    @staticmethod
    def field_extract(value, field):
        """Pull a ``(shift, width)`` field out of a word already in hand."""
        shift, width = field
        return (value >> shift) & ((1 << width) - 1)

    @staticmethod
    def field_insert(value, field, new):
        """Return ``value`` with a field replaced by ``new`` (masked to the field width)."""
        shift, width = field
        mask = ((1 << width) - 1) << shift
        return (value & ~mask) | ((new << shift) & mask)

    @staticmethod
    def bit_reverse(value, bits=8):
        """Reverse the bit order of a ``bits``-wide value (reflected CRCs, LSB-first parts)."""
        out = 0
        for _ in range(bits):
            out = (out << 1) | (value & 1)
            value >>= 1
        return out

    @staticmethod
    def bytes_to_int(data, big_endian=True):
        """Assemble bytes into an unsigned int, MSB first unless ``big_endian`` is False."""
        return int.from_bytes(data, "big" if big_endian else "little")

    @staticmethod
    def int_to_bytes(value, nbytes, big_endian=True):
        """Split ``value`` into ``nbytes`` bytes, masking to width first.

        Masking rather than raising means an oversized argument is truncated instead of
        producing a wrong-length transfer.
        """
        value &= (1 << (8 * nbytes)) - 1
        return value.to_bytes(nbytes, "big" if big_endian else "little")

### validation/test: `sign_extend`
The TMP102 is the worked example: its temperature register is left-justified, 12 bits in
normal mode and 13 bits in extended mode (`EM`), with 0.0625 °C per count. The padding bits
must be shifted out before sign-extending.

#### validation: nominal usage
A TMP102 reading of `0xE700` in normal mode: shift out the 4 padding bits, then extend 12
bits. $-400 \times 0.0625 = -25$ °C.

In [ ]:
word = 0xE700
counts = BitTools.sign_extend(word >> 4, 12)
print("word {:#06x} -> counts {} -> {} C".format(word, counts, counts * 0.0625))
for bits in (8, 12, 16):
    top = 1 << (bits - 1)
    print("{:>2} bits: max {:>6} | min {:>7} | all ones {}".format(
        bits, BitTools.sign_extend(top - 1, bits), BitTools.sign_extend(top, bits),
        BitTools.sign_extend((1 << bits) - 1, bits)))  # fmt: skip

#### validation: anti-nominal usage
`sign_extend` raises nothing: it masks. The anti-nominal case is the one the docstring warns
about, extending the unshifted word, which gives a wrong value silently.

In [ ]:
wrong = BitTools.sign_extend(0xE700, 12)
print("unshifted 0xE700 as 12 bits:", wrong, "(should be -400; the padding was read as magnitude)")

#### unit

In [ ]:
%%extract ../tests/test_bits.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize("bits", [8, 12, 13, 16, 24])
def test_sign_extend_boundaries(bits):
    hi = (1 << (bits - 1)) - 1
    assert BitTools.sign_extend(hi, bits) == hi
    assert BitTools.sign_extend(1 << (bits - 1), bits) == -(1 << (bits - 1))
    assert BitTools.sign_extend((1 << bits) - 1, bits) == -1
    assert BitTools.sign_extend(0, bits) == 0

In [ ]:
%%extract ../tests/test_bits.py -a --strip-ipytest
%%ipytest


def test_sign_extend_masks_oversized_input():
    assert BitTools.sign_extend(0xFFFFF, 12) == BitTools.sign_extend(0xFFF, 12) == -1

In [ ]:
%%extract ../tests/test_bits.py -a --strip-ipytest
%%ipytest


def test_sign_extend_exhaustive_8_bit():
    for v in range(256):
        assert BitTools.sign_extend(v, 8) == (v - 256 if v & 0x80 else v)

In [ ]:
%%extract ../tests/test_bits.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize(
    "word, em, counts",
    [(0xE700, False, -400), (0x3200, False, 800), (0xF380, True, -400), (0x3200, True, 1600)],
)
def test_sign_extend_after_shift_tmp102(word, em, counts):
    shift, bits = (3, 13) if em else (4, 12)
    assert BitTools.sign_extend(word >> shift, bits) == counts

In [ ]:
%%extract ../tests/test_bits.py -a --strip-ipytest
%%ipytest


def test_sign_extend_unshifted_is_wrong():
    assert BitTools.sign_extend(0xE700, 12) != -400

### validation/test: bitfields
The TMP102 configuration register again: nine fields that tile bits 15..4 exactly, and its
power-on value `0x60A0` (`RES` = `0b11`, `CR` = `0b10`, `AL` = 1). The table is shared by the
validation and the tests, so it is written to the test file too.

In [ ]:
%%extract ../tests/test_bits.py -a
TMP102_FIELDS = (
    ("OS", (15, 1), 0x8000),
    ("RES", (13, 2), 0x6000),
    ("FQ", (11, 2), 0x1800),
    ("POL", (10, 1), 0x0400),
    ("TM", (9, 1), 0x0200),
    ("SD", (8, 1), 0x0100),
    ("CR", (6, 2), 0x00C0),
    ("AL", (5, 1), 0x0020),
    ("EM", (4, 1), 0x0010),
)
CFG_POR = 0x60A0

#### validation: nominal usage
Every field's mask and its value in the power-on word; then set the conversion rate to 8 Hz
(`CR` = `0b11`) and read it back.

In [ ]:
for name, field, _ in TMP102_FIELDS:
    print("{:<4} {!s:<8} mask {:#06x} | POR value {}".format(
        name, field, BitTools.field_mask(field), BitTools.field_extract(CFG_POR, field)))  # fmt: skip
cfg = BitTools.field_insert(CFG_POR, (6, 2), 0b11)
print("CR -> 0b11: {:#06x} -> {:#06x} | CR now {}".format(CFG_POR, cfg, BitTools.field_extract(cfg, (6, 2))))

#### validation: anti-nominal usage
None raises: a value too wide for its field is masked, so it cannot spill into the next
field. Writing `0xFF` into the 1-bit `SD` field sets only bit 8.

In [ ]:
print("SD <- 0xFF: {:#06x}".format(BitTools.field_insert(0x0000, (8, 1), 0xFF)))

#### unit

In [ ]:
%%extract ../tests/test_bits.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize("name, field, mask", TMP102_FIELDS)
def test_field_mask(name, field, mask):
    assert BitTools.field_mask(field) == mask

In [ ]:
%%extract ../tests/test_bits.py -a --strip-ipytest
%%ipytest


def test_fields_tile_without_overlap():
    covered = 0
    for _, field, _ in TMP102_FIELDS:
        m = BitTools.field_mask(field)
        assert covered & m == 0
        covered |= m
    assert covered == 0xFFF0

In [ ]:
%%extract ../tests/test_bits.py -a --strip-ipytest
%%ipytest


def test_field_extract_por():
    assert BitTools.field_extract(CFG_POR, (13, 2)) == 0b11
    assert BitTools.field_extract(CFG_POR, (6, 2)) == 0b10
    assert BitTools.field_extract(CFG_POR, (5, 1)) == 1

In [ ]:
%%extract ../tests/test_bits.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize("name, field, mask", TMP102_FIELDS)
def test_field_insert_round_trip_and_isolation(name, field, mask):
    for v in range(1 << field[1]):
        out = BitTools.field_insert(CFG_POR, field, v)
        assert BitTools.field_extract(out, field) == v
        assert out & ~mask == CFG_POR & ~mask

In [ ]:
%%extract ../tests/test_bits.py -a --strip-ipytest
%%ipytest


def test_field_insert_clamps():
    assert BitTools.field_insert(0x0000, (8, 1), 0xFF) == 0x0100

### validation/test: `bit_reverse`
Used by reflected CRCs (`CRCMixin` with `CRC_REFLECT_IN`/`OUT`) and LSB-first parts.

#### validation: nominal usage

In [ ]:
for value, bits in ((0x80, 8), (0xD2, 8), (0x0001, 16)):
    print("{:0{}b} -> {:0{}b}".format(value, bits, BitTools.bit_reverse(value, bits), bits))

#### validation: anti-nominal usage
None: `bit_reverse` has no failure path. Bits above `bits` are ignored.

In [ ]:
print("0x1FF as 8 bits -> {:#04x}".format(BitTools.bit_reverse(0x1FF, 8)))

#### unit

In [ ]:
%%extract ../tests/test_bits.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize(
    "value, bits, want",
    [(0x80, 8, 0x01), (0x01, 8, 0x80), (0xD2, 8, 0x4B), (0x0001, 16, 0x8000),
     (0x8000, 16, 0x0001), (0xFFFF, 16, 0xFFFF), (0, 8, 0), (0x99, 8, 0x99)],
)  # fmt: skip
def test_bit_reverse(value, bits, want):
    assert BitTools.bit_reverse(value, bits) == want

In [ ]:
%%extract ../tests/test_bits.py -a --strip-ipytest
%%ipytest


def test_bit_reverse_involution():
    assert all(BitTools.bit_reverse(BitTools.bit_reverse(v, 8), 8) == v for v in range(256))

### validation/test: bytes and ints
`bytes_to_int` and `int_to_bytes` are thin wrappers on `int.from_bytes` / `int.to_bytes`
with a `big_endian` flag (most I2C parts send MSB first) and, for `int_to_bytes`, masking.

#### validation: nominal usage

In [ ]:
for data in (b"\x12\x34", b"\x12\x34\x56"):
    print("{} -> big {:#x} | little {:#x}".format(
        data, BitTools.bytes_to_int(data), BitTools.bytes_to_int(data, False)))  # fmt: skip
print("0x1234 in 2 bytes -> big {} | little {}".format(
    BitTools.int_to_bytes(0x1234, 2), BitTools.int_to_bytes(0x1234, 2, False)))  # fmt: skip

#### validation: anti-nominal usage
`int_to_bytes` does not raise on a value too big for `nbytes` (or a negative one): it keeps
the low bytes, so the transfer always has the length asked for.

In [ ]:
print("0xABCDEF in 2 bytes -> {} | -1 in 1 byte -> {}".format(
    BitTools.int_to_bytes(0xABCDEF, 2), BitTools.int_to_bytes(-1, 1)))  # fmt: skip

#### unit

In [ ]:
%%extract ../tests/test_bits.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize(
    "data, be, want",
    [(b"\x12\x34", True, 0x1234), (b"\x12\x34", False, 0x3412), (b"\xff", True, 0xFF),
     (b"\x12\x34\x56", True, 0x123456), (b"\x12\x34\x56", False, 0x563412),
     (b"\x00\x00", True, 0)],
)  # fmt: skip
def test_bytes_to_int(data, be, want):
    assert BitTools.bytes_to_int(data, be) == want

In [ ]:
%%extract ../tests/test_bits.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize(
    "value, n, be, want",
    [(0x1234, 2, True, b"\x12\x34"), (0x1234, 2, False, b"\x34\x12"), (0xFF, 1, True, b"\xff"),
     (0x123456, 3, True, b"\x12\x34\x56"), (0x123456, 3, False, b"\x56\x34\x12")],
)  # fmt: skip
def test_int_to_bytes(value, n, be, want):
    assert BitTools.int_to_bytes(value, n, be) == want

In [ ]:
%%extract ../tests/test_bits.py -a --strip-ipytest
%%ipytest


def test_int_to_bytes_truncates():
    assert BitTools.int_to_bytes(0xABCDEF, 2) == b"\xcd\xef"
    assert BitTools.int_to_bytes(-1, 1) == b"\xff"

In [ ]:
%%extract ../tests/test_bits.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize("n", [1, 2, 3, 4])
@pytest.mark.parametrize("be", [True, False])
def test_byte_round_trip(n, be):
    for value in (0, 1, (1 << (8 * n)) - 1, (1 << (8 * n)) // 3):
        assert BitTools.bytes_to_int(BitTools.int_to_bytes(value, n, be), be) == value

# Hardware validation

> ## ⚠️ STOP — do not run this section unattended
>
> **Everything below talks to the real board.** It connects to the FeatherS3 and uploads the
> package. Nothing here uses the I2C bus.
>
> - Run it **one cell at a time**. Read the **What to do** line above each cell first.
> - The section ends with `%board disconnect`. If you stop partway, run that cell yourself.

Layout: connect and set up, the **matching tests** (the same vectors as the unit tests
above, on MicroPython's integers and `int.from_bytes`/`to_bytes`), then the tally and
disconnect.

## Connect and set up

Notebook-only: connect to the board. Only one notebook can own the board: if another notebook
still has it, run `%board disconnect` there first.

**What to do:** make sure the board is plugged in and no other notebook owns it.

In [ ]:
%board connect

Notebook-only: remove the package an earlier upload left on the board, and make sure `/lib`
exists, so the copy below lands exactly at `/lib/micropython_i2c`.

**What to do:** nothing.

In [ ]:
import os


def rmtree(path):
    """Delete a board folder and everything in it."""
    for entry in os.ilistdir(path):
        child = path + "/" + entry[0]
        if entry[1] == 0x4000:  # folder
            rmtree(child)
        else:
            os.remove(child)
    os.rmdir(path)


try:
    rmtree("/lib/micropython_i2c")
    print("removed the old /lib/micropython_i2c")
except OSError:
    print("no /lib/micropython_i2c on the board yet")
try:
    os.mkdir("/lib")
except OSError:
    pass  # already there

Notebook-only: copy `src/micropython_i2c`, including the file this notebook just wrote, to
`/lib/micropython_i2c`. The copy runs on the host; the path is relative to this notebook's
folder.

**What to do:** nothing. Takes a few seconds.

In [ ]:
%%host
from pathlib import Path

pkg = next(p for p in (Path("../src"), Path("src")) if p.is_dir()) / "micropython_i2c"
%files copy -r {pkg} :/lib/micropython_i2c

Notebook-only: drop any modules an earlier run left on the board, so the new files load, then
import what the checks use and zero the tally. The TMP102 field table is redefined here: the
board has not seen the host's.

**What to do:** nothing.

In [ ]:
import sys

for name in [m for m in sys.modules if m.startswith("micropython_i2c")]:
    del sys.modules[name]

from micropython_i2c.bits import BitTools
from micropython_i2c.debug import check, check_reset, check_summary
from micropython_i2c.debug import check_counts  # noqa: F401  (read from the host with %eval)

TMP102_FIELDS = (
    ("OS", (15, 1), 0x8000),
    ("RES", (13, 2), 0x6000),
    ("FQ", (11, 2), 0x1800),
    ("POL", (10, 1), 0x0400),
    ("TM", (9, 1), 0x0200),
    ("SD", (8, 1), 0x0100),
    ("CR", (6, 2), 0x00C0),
    ("AL", (5, 1), 0x0020),
    ("EM", (4, 1), 0x0010),
)
CFG_POR = 0x60A0
check_reset()

## Matching tests

**Matches** `test_sign_extend_boundaries`, `test_sign_extend_masks_oversized_input` and
`test_sign_extend_exhaustive_8_bit`.

**What to do:** nothing.

In [ ]:
for bits in (8, 12, 13, 16, 24):
    hi = (1 << (bits - 1)) - 1
    ok = (BitTools.sign_extend(hi, bits) == hi
          and BitTools.sign_extend(1 << (bits - 1), bits) == -(1 << (bits - 1))
          and BitTools.sign_extend((1 << bits) - 1, bits) == -1
          and BitTools.sign_extend(0, bits) == 0)  # fmt: skip
    check("test_sign_extend_boundaries[{}]".format(bits), ok)
check("test_sign_extend_masks_oversized_input",
      BitTools.sign_extend(0xFFFFF, 12) == BitTools.sign_extend(0xFFF, 12) == -1)  # fmt: skip
check("test_sign_extend_exhaustive_8_bit",
      all(BitTools.sign_extend(v, 8) == (v - 256 if v & 0x80 else v) for v in range(256)))  # fmt: skip

**Matches** `test_sign_extend_after_shift_tmp102` and `test_sign_extend_unshifted_is_wrong`.

**What to do:** nothing.

In [ ]:
for word, em, counts in ((0xE700, False, -400), (0x3200, False, 800),
                         (0xF380, True, -400), (0x3200, True, 1600)):  # fmt: skip
    shift, bits = (3, 13) if em else (4, 12)
    got = BitTools.sign_extend(word >> shift, bits)
    check("test_sign_extend_after_shift_tmp102[{:#06x}]".format(word), got == counts,
          "{} counts".format(got))  # fmt: skip
check("test_sign_extend_unshifted_is_wrong", BitTools.sign_extend(0xE700, 12) != -400)

**Matches** `test_field_mask`, `test_fields_tile_without_overlap`, `test_field_extract_por`,
`test_field_insert_round_trip_and_isolation` and `test_field_insert_clamps`.

**What to do:** nothing.

In [ ]:
check("test_field_mask", all(BitTools.field_mask(f) == m for _, f, m in TMP102_FIELDS))
covered, overlap = 0, False
for _, f, _ in TMP102_FIELDS:
    m = BitTools.field_mask(f)
    overlap = overlap or bool(covered & m)
    covered |= m
check("test_fields_tile_without_overlap", not overlap and covered == 0xFFF0,
      "{:#06x}".format(covered))  # fmt: skip
check("test_field_extract_por",
      BitTools.field_extract(CFG_POR, (13, 2)) == 0b11
      and BitTools.field_extract(CFG_POR, (6, 2)) == 0b10
      and BitTools.field_extract(CFG_POR, (5, 1)) == 1)  # fmt: skip
ok = True
for _, f, m in TMP102_FIELDS:
    for v in range(1 << f[1]):
        out = BitTools.field_insert(CFG_POR, f, v)
        ok = ok and BitTools.field_extract(out, f) == v and out & ~m == CFG_POR & ~m
check("test_field_insert_round_trip_and_isolation", ok)
check("test_field_insert_clamps", BitTools.field_insert(0x0000, (8, 1), 0xFF) == 0x0100)

**Matches** `test_bit_reverse` and `test_bit_reverse_involution`.

**What to do:** nothing.

In [ ]:
cases = ((0x80, 8, 0x01), (0x01, 8, 0x80), (0xD2, 8, 0x4B), (0x0001, 16, 0x8000),
         (0x8000, 16, 0x0001), (0xFFFF, 16, 0xFFFF), (0, 8, 0), (0x99, 8, 0x99))  # fmt: skip
check("test_bit_reverse", all(BitTools.bit_reverse(v, b) == w for v, b, w in cases))
check("test_bit_reverse_involution",
      all(BitTools.bit_reverse(BitTools.bit_reverse(v, 8), 8) == v for v in range(256)))  # fmt: skip

**Matches** `test_bytes_to_int`, `test_int_to_bytes`, `test_int_to_bytes_truncates` and
`test_byte_round_trip`: MicroPython's `int.from_bytes` and `int.to_bytes` with both byte
orders.

**What to do:** nothing.

In [ ]:
cases = ((b"\x12\x34", True, 0x1234), (b"\x12\x34", False, 0x3412), (b"\xff", True, 0xFF),
         (b"\x12\x34\x56", True, 0x123456), (b"\x12\x34\x56", False, 0x563412),
         (b"\x00\x00", True, 0))  # fmt: skip
check("test_bytes_to_int", all(BitTools.bytes_to_int(d, be) == w for d, be, w in cases))
cases = ((0x1234, 2, True, b"\x12\x34"), (0x1234, 2, False, b"\x34\x12"), (0xFF, 1, True, b"\xff"),
         (0x123456, 3, True, b"\x12\x34\x56"), (0x123456, 3, False, b"\x56\x34\x12"))  # fmt: skip
check("test_int_to_bytes", all(BitTools.int_to_bytes(v, n, be) == w for v, n, be, w in cases))
check("test_int_to_bytes_truncates",
      BitTools.int_to_bytes(0xABCDEF, 2) == b"\xcd\xef" and BitTools.int_to_bytes(-1, 1) == b"\xff")  # fmt: skip
ok = True
for n in (1, 2, 3, 4):
    for be in (True, False):
        for value in (0, 1, (1 << (8 * n)) - 1, (1 << (8 * n)) // 3):
            ok = ok and BitTools.bytes_to_int(BitTools.int_to_bytes(value, n, be), be) == value
check("test_byte_round_trip", ok)

## Tally and disconnect

Notebook-only: the PASS/FAIL tally of every check in this section.

**What to do:** read it. Any `FAIL` line above says which check and why.

In [ ]:
check_summary()

Notebook-only: copy the tally to the host before disconnecting, so the last cell can fail the
run after the board is released.

**What to do:** nothing.

In [ ]:
%%host
hw_passed, hw_failed = get_ipython().run_line_magic("eval", "check_counts()")

Notebook-only: release the board, so the next run (or a kernel restart) starts on the host.

**What to do:** nothing.

In [ ]:
%board disconnect

Notebook-only: fail the run if any hardware check failed. It runs on the host.

**What to do:** nothing. An error here means a `FAIL` line above.

In [ ]:
%%host
if hw_failed:
    raise AssertionError("{} of {} hardware checks failed".format(hw_failed, hw_passed + hw_failed))
print("all {} hardware checks passed".format(hw_passed))